# 02 - Chunking
## Shakespeare Evidence Desk


## Why This Stage

*In general — what chunking is and why any Retrieval-Augmented Generation (RAG) system needs it.* Retrieval systems never operate on whole documents — they operate on pieces. A chunk is the unit that gets embedded, indexed, retrieved, and ultimately quoted back as evidence, so its size and boundaries quietly set the ceiling for the entire system. Chunks that are too large bundle several ideas into one vector, so a search returns a sprawling passage where only a fraction was relevant. Chunks that are too small split a single idea across fragments, so the piece that matches a query no longer contains the whole answer.

Chunking sits between parsing and embedding: it needs the structure parsing produced, but must finish before anything is encoded into vectors. It is one of the highest-leverage and most underestimated choices in a RAG pipeline — every downstream measure, from whether the right passage can be retrieved at all to whether it can be cited precisely, is bounded by how the text was divided here. That is why the chunking strategy is compared and measured rather than picked by feel.


- **Goal:** split the parsed blocks from notebook 01 into retrieval units that are large enough to answer a question, small enough to retrieve precisely, and never silently merge two speakers or cross a scene boundary without recording it.
- **Loads from Step 01:** `.artifacts/parsed-blocks.json` (the structured, source-linked blocks) and `.artifacts/parsing-handoff.json` (the parser decision). The labeled questions (`.artifacts/eval-questions.json`) drive the answerability measure.
- **The experiment:** compare four deterministic strategies — **fixed-size**, **fixed-size + overlap**, **structure-aware recursive packing**, and **scene-aware** — then reserve a **model-assisted** arm as a cost-bounded fallback, mirroring the routing policy chosen in notebook 01.
- **Runs offline:** this notebook only reads local artifacts; the only network call is the optional model-assisted demo, which skips gracefully if Azure is unavailable.
- **Preserve provenance:** every chunk keeps its work id, act, scene, speakers, and source-line span so later stages and labels can trace back to the original text.
- **Not later stages:** this notebook does not embed, index, retrieve, or answer. It exports selected chunks for notebook 03 (embeddings).


## Conceptual Foundations

### The chunk is the atomic unit of retrieval

Everything downstream operates on chunks, not documents: a chunk is what gets embedded (03), stored as an index row (04), retrieved (05), and quoted as evidence (08). So the chunk boundary silently decides what *can* be retrieved at all. If the sentence that answers a question is split across two chunks, no retriever can return it whole; if it's buried inside a 3,000-token chunk full of unrelated material, the match is diluted and may rank below weaker-but-tighter chunks.

### The central tension: precision vs. recall of a single chunk

Chunk size trades two failures against each other:

- **Chunks too large** → high *recall* (the answer is probably in there somewhere) but low *precision* (the vector represents an average of several ideas, so it matches weakly, and the model must wade through noise). This is the classic failure of the **scene-aware** strategy below.
- **Chunks too small** → high *precision* per chunk but low *recall* (a single idea is fragmented, so the matching piece no longer contains the whole answer). This is the failure of blind small windows.

Good chunking finds the middle: **self-contained, attributable units of evidence** — big enough to answer, small enough to retrieve precisely, and never silently merging two speakers or crossing a scene without recording it.

### The two knobs: size and overlap

- **Target size (512 tokens here)** — the budget each chunk aims for. Measured in **tokens**, not characters or words, because that is what the embedding model actually counts (and has a hard input limit on). A token is roughly ¾ of a word; 512 tokens ≈ 350–400 words ≈ a healthy paragraph or two.
- **Overlap (64 tokens here)** — how much text each chunk repeats from its predecessor. Overlap is "seam glue": it rescues a sentence that straddles a boundary by making it appear whole in at least one chunk. Too little loses context at the seams; too much duplicates text and inflates the index.

### Structure-blind vs. structure-aware

The deepest distinction among strategies is whether they *see* the document's structure. **Structure-blind** methods (fixed windows) treat text as one undifferentiated stream and cut on a token count — simple, but they happily slice mid-speech or jam two scenes together. **Structure-aware** methods cut on natural seams the parser exposed (speech, scene, act), so a chunk is always a clean dramatic unit. Because notebook 01 preserved that structure, we can afford to be structure-aware — a concrete payoff of the earlier stage's provenance work.

### Common misconceptions and failure modes

- **"Bigger chunks are safer because they contain more."** They raise recall but wreck precision and can overflow the embedding window — the single most common chunking mistake.
- **"Overlap fixes bad boundaries."** It softens seams; it does not repair a cut that split an idea in half. Structure-aware boundaries do that.
- **"One chunk size is universally best."** Optimal size depends on the question type (pinpoint fact vs. summary), the embedding model's window, and the corpus style. It is a measured choice, re-tuned once retrieval is live.
- **"Chunking can be validated here."** The honest metrics at this stage are proxies (size distribution, boundary integrity). Whether chunking *helps the final answer* is only knowable end-to-end in notebooks 05 and 10.


## 1. Load the Parsed Blocks and Labeled Questions

- **Purpose:** start from notebook 01's exported blocks and decision record instead of hidden kernel state. The blocks are already clean and source-linked, so chunking is a structure-aware operation, not raw byte slicing.
- **What the code does:** locates the repository root (using `.env.example` as a marker), reads `parsed-blocks.json`, groups the blocks by work in reading order, and loads the labeled questions that the answerability measure (Step 4) will use.
- **Fails loudly if missing:** if `.artifacts/parsed-blocks.json` is absent, run notebook 01 first — this notebook cannot invent the corpus structure.
- **Expected result:** the corpus prefix, the total block count across the ten plays, the block-type breakdown (speech / scene_heading / stage_direction), and the development/held-out question counts. No network call happens here.


In [ ]:
import json
import re
from collections import Counter, defaultdict  # Counter: tallies counts; defaultdict: dict with auto-created values
from pathlib import Path  # Path: object-oriented file paths (the `/` operator joins path parts)

# Path.cwd() = current working directory; .resolve() turns it into a full absolute path.
ROOT = Path.cwd().resolve()
# .is_file() tests whether a path exists and is a file. If .env.example isn't here, try the parent folder.
if not (ROOT / '.env.example').is_file() and (ROOT.parent / '.env.example').is_file():
    ROOT = ROOT.parent
if not (ROOT / '.env.example').is_file():
    raise RuntimeError('Use the repository as the notebook working directory.')

ARTIFACTS = ROOT / '.artifacts'  # `/` joins paths: ROOT/.artifacts
blocks_path = ARTIFACTS / 'parsed-blocks.json'
parsing_handoff_path = ARTIFACTS / 'parsing-handoff.json'
questions_path = ARTIFACTS / 'eval-questions.json'
if not blocks_path.is_file():
    raise RuntimeError('Missing .artifacts/parsed-blocks.json. Run notebook 01 first.')

# .read_text() reads the whole file as a string; json.loads() parses that JSON string into Python dicts/lists.
parsed = json.loads(blocks_path.read_text(encoding='utf-8'))
all_blocks = parsed['blocks']
corpus_prefix = parsed['corpus_prefix']
# Load optional files only if present; otherwise fall back to an empty default (the `... if ... else ...` expression).
parsing_handoff = json.loads(parsing_handoff_path.read_text(encoding='utf-8')) if parsing_handoff_path.is_file() else {}
evaluation_questions = json.loads(questions_path.read_text(encoding='utf-8')) if questions_path.is_file() else {'development': [], 'heldout': []}

# defaultdict(list) gives every new key an empty list automatically, so we can .append without checking first.
blocks_by_work = defaultdict(list)
for block in all_blocks:
    blocks_by_work[block['work_id']].append(block)

# Dict comprehension: {key: value for ...} builds a lookup from work id -> play title.
work_titles = {block['work_id']: block['work'] for block in all_blocks}
# Counter counts how many blocks of each type exist, e.g. Counter({'speech': 8159, ...}).
type_counts = Counter(block['type'] for block in all_blocks)

print(f'Corpus prefix: {corpus_prefix}')
print(f'Loaded {len(all_blocks):,} blocks across {len(blocks_by_work)} works.')  # :, adds thousands separators
print('Block types: ' + ', '.join(f'{name}={count}' for name, count in sorted(type_counts.items())))
# dict.get(key, default) reads a key safely, returning the default if it's missing.
print(f'Questions: {len(evaluation_questions.get("development", []))} development, '
      f'{len(evaluation_questions.get("heldout", []))} held-out.')


## 2. Explain the Alternatives

A chunk is "correct" for this corpus when it is a **self-contained, attributable unit of evidence**. That turns into three measurable properties: a controlled **size distribution** (fits the embedding model's window, few tiny or oversized chunks), **provenance integrity** (never fragments a scene across the index or merges unrelated scenes), and **answerability** (the evidence a labeled question needs lands inside as few chunks as possible). We vary one factor — the boundary rule — while holding the parser, corpus, and token budget fixed.

**In plain terms** — think of the play as a long ribbon of text we must cut into index cards. Each strategy is a different cutting rule:

| Strategy | Plain-terms idea | Expected failure mode |
|---|---|---|
| **fixed-512** | Cut every 512 tokens, blindly — ignore where speeches or scenes end. Baseline. | Cuts speeches mid-sentence and packs unrelated scenes together; loses speaker attribution. |
| **overlap-512-64** | Same blind cut, but each card repeats the previous card's last ~64 tokens so a straddling sentence survives on one card. | Recovers boundary context but duplicates text and inflates the index. |
| **recursive-512-64** | Cut on natural seams (whole blocks), pack them up to ~512 tokens, and **never span two scenes**; hard-split any single oversized block with overlap. | Uniform sizes, but a long scene still spans several chunks. |
| **scene-aware** | One card per whole scene (the natural dramatic unit); split only scenes above a hard max. | Keeps a scene whole but produces large, uneven chunks that can dilute retrieval precision. |
| **model-assisted** *(reserved)* | Ask the chat deployment to propose semantic boundaries for a scene. | Real latency and token cost, non-deterministic — reserved as a targeted fallback, not the default. |

- **Why 512 and 64?** **512 tokens** (~350–400 words) is the sweet spot: big enough to hold a coherent passage with context, small enough to retrieve *precisely*, and comfortably inside the embedding model's input limit (notebook 03) so nothing gets truncated. **64-token overlap** (~12%) is the "seam glue" — repeating a little text across a cut keeps a sentence that straddles the boundary readable on at least one card. Too little loses context; too much bloats the index. Both are fixed knobs so we compare boundary *rules* fairly, one change at a time.
- **Library equivalence:** the fixed and recursive strategies are implemented inline rather than via LangChain's `RecursiveCharacterTextSplitter`, to keep the notebook transparent and dependency-light; the behaviour is the same, but here every chunk carries full provenance.
- **Honest expectation:** there is no free ground truth for "best chunk." We report distribution, integrity, and an answerability proxy against the labeled scenes, then pick the simplest strategy that meets the constraints.


### The Wider Landscape (and where this notebook sits in it)

![Overview of chunking strategies](images/chunking-strategies-overview.png)
*Source: [DataCamp — Chunking Strategies for AI and RAG Applications](https://www.datacamp.com/blog/chunking-strategies) ([image](https://media.datacamp.com/cms/d47e07c42a1e601fb77b5847b7540a6d.png))*

![The 7 chunking strategies feeding a vector database](images/chunking-pipeline.png)
*Source: [Machine Learning Mastery — 7 Chunking Strategies That Decide Whether Your RAG Works](https://machinelearningmastery.com/7-chunking-strategies-that-decide-whether-your-rag-works/) ([image](https://machinelearningmastery.com/wp-content/uploads/2026/08/mlm-chugani-7-chunking-strategies-decide-rag-works-1.png))*

The diagrams above show the full menu of chunking strategies. We deliberately implement a **focused subset** that fits this corpus (plain, well-structured dramatic text) and this pipeline stage. The rest are either retrieval-time architectures or for data types we don't have:

| Strategy (from the references) | In this notebook? | Where it belongs |
|---|---|---|
| Fixed-size (+ overlap / sliding-window) | ✅ `fixed-512`, `overlap-512-64` | here — the baseline |
| Recursive / document-aware structural | ✅ `recursive-512-64`, `scene-aware` | here — our winner cuts on act/scene/speech seams |
| Semantic (embedding-drift) | ◐ reserved | needs per-sentence embeddings → cost; cousin of our model arm |
| Agentic / AI-driven dynamic | ◐ demoed on one scene | reserved fallback (non-deterministic, costly) |
| Sentence-window (small-to-big) | ✖ | **retrieval** architecture → notebook 05 |
| Hierarchical / parent-child | ✖ | **indexing + retrieval** → notebooks 04–05 |
| Topic-based (LDA / clustering) | ✖ | useful for unstructured prose; our structure already groups topics by scene |
| Modality / table-preserving | ✖ | N/A — no tables, charts, or images in plays |
| Late / post-chunking | ✖ | alternate ordering (embed whole doc, chunk at query time) → revisit with long-context retrieval |

**Two things from the references worth carrying forward:**

- **"Contextual chunking" = enrich chunks with metadata for traceability.** The guides call this out as a best practice for regulated/evidence domains — which validates the provenance (`work/act/scene/speaker/lines`) we attach to every chunk. That was the right call, not over-decoration.
- **The real scorecards are retrieval-level, measured later.** The metrics that actually decide whether chunking "works" — **context precision**, **context recall**, **context relevancy**, **chunk utilization** (how much of a retrieved chunk the model used), and **chunk attribution** (which chunks produced the answer) — all require a live retriever and generator. They land in notebooks 05 (retrieval) and 10 (continuous evaluation). Our local proxies here only pre-screen; these are the end-to-end judges we'll use when we revisit this choice.

> Pipeline position (both references agree): chunking happens **after** parsing/preprocessing and **before** embedding — exactly our `01 → 02 → 03` order. The chunk text becomes a vector in notebook 03 and a searchable row (vector + metadata) in the vector database in notebook 04.


### How to Choose a Chunking Strategy, and the Tools That Implement Them

#### A decision guide

Chunking choice follows from two questions: *how structured is your text?* and *what kind of questions will you ask?*

```
Does the document expose reliable structure (headings, sections, speakers)?
├─ No (flat prose, logs, transcripts)
│     ├─ Questions are pinpoint facts  → small fixed/recursive windows with overlap
│     └─ Questions are broad/summary    → larger windows, or semantic chunking
└─ Yes (our plays, Markdown, HTML, code)
      ├─ Need precise citation + tight retrieval → structure-aware recursive packing  ← our pick
      ├─ Evidence must never be fragmented        → section/scene-aware (watch the size blow-up)
      └─ Long docs + both precise & broad queries → hierarchical / parent-child (retrieval-time)
```

**Rules of thumb:**
- Start with **recursive, structure-aware packing at ~256–512 tokens with small overlap** — it is the robust default for most corpora and the one this notebook selects.
- Only reach for **semantic chunking** (cut where embedding similarity drops) when structure is absent *and* the extra per-sentence embedding cost is justified.
- Treat **size and overlap as tunables**, not constants — revisit them once retrieval (05) can measure their real effect.
- Match size to the **embedding window and question type**: smaller for fact lookup, larger for synthesis.

#### Where each strategy's quality actually shows up

The proxies measured in this notebook (size distribution, scene-boundary integrity, fragmentation) only *pre-screen*. The metrics that truly judge chunking are **retrieval-level** and are measured later: context precision/recall (05), and chunk utilization and attribution (10). Choose here on principle and proxies; confirm downstream with real retrieval.

#### Tools and libraries

We implement the splitters inline (plain Python + `tiktoken`) so every boundary rule and every provenance field is visible. Production systems commonly use:

- **LangChain** — `RecursiveCharacterTextSplitter` (our `recursive` strategy's library equivalent), `CharacterTextSplitter`, `TokenTextSplitter`, and Markdown/HTML/code-aware splitters.
- **LlamaIndex** — node parsers: `SentenceSplitter`, `SentenceWindowNodeParser` (small-to-big), and `HierarchicalNodeParser` (parent-child).
- **Unstructured.io** — "chunk by title"/element-aware chunking that reuses the structure its parser detected.
- **`tiktoken`** — the tokenizer that makes a *token* budget (not a character budget) accurate for OpenAI models; essential for respecting the embedding window.
- **Semantic chunkers** — LangChain's `SemanticChunker` and LlamaIndex's `SemanticSplitterNodeParser` cut where consecutive-sentence embedding similarity drops.

**The engineering point:** these libraries save boilerplate, but they implement the *same* boundary rules taught here. Understanding the rule matters more than the API — and note that most off-the-shelf splitters do **not** carry rich provenance (`work/act/scene/speaker/lines`) by default, which for an evidence-citing system you must add yourself.


## 3. Run Controlled Experiments

- **Fixed upstream:** the parser output, corpus snapshot, token budget (512), and overlap (64) are held constant; only the boundary rule changes between strategies.
- **What the code does:** sets up a tokenizer (with a safe fallback), turns each block into a provenance-carrying "unit" (assembled text plus work / act / scene / speaker / source-line span), then runs all four deterministic strategies across the ten plays.
- **Provenance is first-class:** every chunk records the exact set of `(act, scene)` pairs it covers, its speakers, and its source-line span, so the integrity and answerability measures in Step 4 are computed from the chunks themselves, not re-parsed.
- **Expected result:** a chunk count per strategy. The structure-aware strategies should produce fewer, cleaner chunks than the structure-blind windows.


In [ ]:
TARGET_TOKENS = 512
OVERLAP_TOKENS = 64
SCENE_MAX_TOKENS = 8000  # scene-aware keeps whole scenes up to the embedding model's practical ceiling

# A tokenizer turns text into the integer "tokens" a model counts. tiktoken is OpenAI's tokenizer library.
try:
    import tiktoken

    _encoding = tiktoken.get_encoding('cl100k_base')  # the encoding used by current OpenAI embedding/chat models
    encode = _encoding.encode  # text -> list[int] tokens
    decode = _encoding.decode  # list[int] tokens -> text
    TOKENIZER = 'tiktoken/cl100k_base'
except Exception:
    # Fallback if tiktoken isn't installed: treat each character as one "token" so the notebook still runs.
    def encode(text):
        return list(text)

    def decode(tokens):
        return ''.join(tokens)

    TOKENIZER = 'char-approx'


def count_tokens(text):
    return len(encode(text))


_ROMAN = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}


def roman_to_int(value):
    # Labels cite scenes as arabic ("Scene 5"), the parser stored roman ("V"); normalize both to ints.
    if value is None:
        return None
    text = str(value).strip().upper()  # .strip() removes surrounding spaces; .upper() uppercases
    if text.isdigit():  # .isdigit() is True for strings like "5"
        return int(text)
    # Subtractive notation: a smaller symbol before a larger one subtracts (IV = 4, IX = 9).
    total, prev = 0, 0
    for char in reversed(text):  # reversed() walks the string right-to-left
        if char not in _ROMAN:
            return None
        current = _ROMAN[char]
        total += -current if current < prev else current
        prev = max(prev, current)
    return total or None  # `x or None` returns None when total is 0


def _scene_sort_key(pair):
    # None acts/scenes (front matter) sort before numbered ones.
    act, scene = pair
    return (act is not None, act or 0, scene is not None, scene or 0)


def unit_text(block):
    # A "unit" is one block turned into chunk-ready text, with a tag so speakers/directions stay readable.
    if block['type'] == 'speech':
        return f"{block['speaker']}: {block['text']}"  # f-string inserts the variables into the text
    if block['type'] == 'scene_heading':
        return f"[Scene] {block['text']}"
    return f"[Stage direction] {block['text']}"


def build_units(blocks):
    # Attach everything a chunk later needs: assembled text, provenance (act/scene/speaker/line span), token size.
    units = []
    for block in blocks:
        text = unit_text(block)
        start = block.get('source_line') or 0  # .get returns None if missing; `or 0` then defaults it to 0
        units.append({
            'work_id': block['work_id'], 'work': block['work'],
            'act': block['act'], 'scene': block['scene'], 'type': block['type'],
            'speaker': block.get('speaker'), 'text': text,
            'line_start': start, 'line_end': start + block.get('text', '').count('\n'),  # .count('\n') counts lines
            'tokens': count_tokens(text),
        })
    return units


def make_chunk(work_id, work, strategy, idx, units, text=None):
    # Build one chunk record from a list of units, rolling up their provenance.
    # A set comprehension {...} collects the *distinct* (act, scene) pairs; sorted() orders them for stable output.
    # scene_keys drives the answerability lookup later, so each pair is stored as integers.
    scene_keys = sorted({(roman_to_int(u['act']), roman_to_int(u['scene'])) for u in units},
                        key=_scene_sort_key)
    speakers = sorted({u['speaker'] for u in units if u.get('speaker')})  # distinct speakers, alphabetized
    acts = {u['act'] for u in units}
    body = text if text is not None else '\n\n'.join(u['text'] for u in units)  # join blocks with blank lines
    starts = [u['line_start'] for u in units if u['line_start']]  # list comprehension with a filter
    ends = [u['line_end'] for u in units if u['line_end']]
    return {
        'chunk_id': f'{work_id}:{strategy}:{idx}',
        'strategy': strategy, 'work_id': work_id, 'work': work,
        # 'mixed' flags a chunk spanning more than one act/scene (only the structure-blind windows do this).
        'act': units[0]['act'] if len(acts) == 1 else 'mixed',
        'scene': units[0]['scene'] if len(scene_keys) == 1 else 'mixed',
        'scenes': [[a, s] for a, s in scene_keys],
        'scene_count': len(scene_keys),
        'speakers': speakers,
        'line_start': min(starts) if starts else None,  # min/max of the line numbers = the chunk's source span
        'line_end': max(ends) if ends else None,
        'token_count': count_tokens(body),
        'text': body,
    }


def scene_groups(units):
    # Split a work's units into consecutive runs sharing the same (act, scene); each run is one scene.
    groups, current, key = [], [], None
    for unit in units:
        unit_key = (unit['act'], unit['scene'])
        if current and unit_key != key:  # the scene changed -> close the current run and start a new one
            groups.append(current)
            current = []
        key = unit_key
        current.append(unit)
    if current:
        groups.append(current)
    return groups


# =====================================================================================
# STRATEGY 1 & 2 — FIXED-SIZE WINDOWS ("fixed-512" with overlap=0, "overlap-512-64" with overlap=64)
#   What it does: ignores the play's structure; glues the whole work into one token
#   stream and cuts it into equal windows, optionally overlapping neighbours.
#   + Advantages: dead simple, fast, predictable uniform sizes, no parsing needed.
#     Overlap rescues a sentence that lands on a cut by repeating it on the next card.
#   - Disadvantages: "structure-blind" — a cut can land mid-speech or jam two scenes
#     into one chunk (hurts citation + precision). Overlap also duplicates text and
#     inflates the index. Best only for flat, unstructured text (logs, plain prose).
# =====================================================================================
def fixed_chunks(work_id, work, units, size, overlap, strategy):
    # Structure-blind: flatten the work into one token stream, then slice fixed windows (with optional overlap).
    # owner[i] records which unit each token came from, so a sliced window traces back to its blocks.
    tokens, owner = [], []
    for index, unit in enumerate(units):  # enumerate() yields (position, item) pairs
        piece = encode(unit['text'] + '\n\n')
        tokens.extend(piece)  # .extend adds every element of a list (unlike .append, which adds one item)
        owner.extend([index] * len(piece))  # [index] * n repeats the index n times
    # step is the stride; with overlap < size, each window re-reads `overlap` tokens from the previous one.
    chunks, position, idx, step = [], 0, 0, max(size - overlap, 1)
    while position < len(tokens):
        window = tokens[position:position + size]  # slice: tokens from `position` up to `position+size`
        if not window:
            break
        covered = [units[j] for j in sorted(set(owner[position:position + size]))]  # distinct blocks in this window
        chunks.append(make_chunk(work_id, work, strategy, idx, covered, text=decode(window).strip()))
        idx += 1
        if position + size >= len(tokens):
            break
        position += step
    return chunks


def split_oversized(unit, size, overlap):
    # A single block longer than the budget: cut it into budget-sized, overlapping pieces.
    tokens = encode(unit['text'])
    pieces, position, step = [], 0, max(size - overlap, 1)
    while position < len(tokens):
        text = decode(tokens[position:position + size]).strip()
        pieces.append({**unit, 'text': text, 'tokens': count_tokens(text)})  # {**unit, ...} copies unit, overriding keys
        if position + size >= len(tokens):
            break
        position += step
    return pieces


# =====================================================================================
# STRATEGY 3 — STRUCTURE-AWARE RECURSIVE PACKING ("recursive-512-64")  [the winner]
#   What it does: respects boundaries. Scene by scene, it packs whole blocks (speeches,
#   directions) into a chunk until the next one would exceed the budget, then starts a
#   fresh chunk. A single oversized block is hard-split with overlap.
#   + Advantages: never cuts mid-speech, never crosses a scene, yet stays near the
#     512-token target so sizes are uniform and embedding-friendly. Keeps full provenance.
#   - Disadvantages: a long scene still spans several chunks (evidence is fragmented, to
#     be reassembled by overlap + retrieval later). Needs the parsed structure to exist.
# =====================================================================================
def recursive_chunks(work_id, work, units, size, overlap, strategy):
    # Work scene by scene so a chunk never spans two scenes; pack whole blocks until the budget is hit.
    chunks, idx = [], 0
    for group in scene_groups(units):
        buffer, buffer_tokens = [], 0  # blocks waiting to become the next chunk, and their running token total
        for unit in group:
            if unit['tokens'] > size:  # block too big to ever fit — flush buffer, then hard-split it
                if buffer:
                    chunks.append(make_chunk(work_id, work, strategy, idx, buffer))
                    idx += 1
                    buffer, buffer_tokens = [], 0
                for piece in split_oversized(unit, size, overlap):
                    chunks.append(make_chunk(work_id, work, strategy, idx, [piece]))
                    idx += 1
                continue  # skip the normal packing for this oversized block
            # Adding this block would overflow the budget: flush what's buffered, then start a new chunk.
            if buffer and buffer_tokens + unit['tokens'] > size:
                chunks.append(make_chunk(work_id, work, strategy, idx, buffer))
                idx += 1
                buffer, buffer_tokens = [], 0
            buffer.append(unit)
            buffer_tokens += unit['tokens']
        if buffer:  # flush the last partial chunk at the end of each scene
            chunks.append(make_chunk(work_id, work, strategy, idx, buffer))
            idx += 1
    return chunks


# =====================================================================================
# STRATEGY 4 — SCENE-AWARE ("scene-aware")
#   What it does: one chunk per whole scene (the natural dramatic unit); only scenes
#   bigger than the embedding ceiling (8000 tokens) fall back to recursive packing.
#   + Advantages: keeps a scene's evidence perfectly together — best "don't fragment"
#     behaviour and the most human-intuitive unit.
#   - Disadvantages: chunk sizes are huge and wildly uneven (a scene can be thousands of
#     tokens), which overflows the embedding window and dilutes retrieval precision —
#     the classic "chunks too big" failure. Good for summary-style questions, poor for
#     pinpoint fact lookup.
# =====================================================================================
def scene_chunks(work_id, work, units, hard_max, overlap, strategy):
    chunks, idx = [], 0
    for group in scene_groups(units):
        # One chunk per whole scene if it fits the ceiling; otherwise fall back to recursive packing.
        if sum(u['tokens'] for u in group) <= hard_max:
            chunks.append(make_chunk(work_id, work, strategy, idx, group))
            idx += 1
        else:
            for piece in recursive_chunks(work_id, work, group, hard_max, overlap, strategy):
                piece['chunk_id'] = f'{work_id}:{strategy}:{idx}'  # renumber so ids stay sequential per work
                chunks.append(piece)
                idx += 1
    return chunks


units_by_work = {work_id: build_units(blocks) for work_id, blocks in blocks_by_work.items()}

# Each runner maps (work_id, title, units) -> list of chunks; same budget, different boundary rule.
# `lambda w, t, u: ...` is a small inline function with three arguments.
strategy_runners = {
    'fixed-512': lambda w, t, u: fixed_chunks(w, t, u, TARGET_TOKENS, 0, 'fixed-512'),
    'overlap-512-64': lambda w, t, u: fixed_chunks(w, t, u, TARGET_TOKENS, OVERLAP_TOKENS, 'overlap-512-64'),
    'recursive-512-64': lambda w, t, u: recursive_chunks(w, t, u, TARGET_TOKENS, OVERLAP_TOKENS, 'recursive-512-64'),
    'scene-aware': lambda w, t, u: scene_chunks(w, t, u, SCENE_MAX_TOKENS, OVERLAP_TOKENS, 'scene-aware'),
}

strategies = {}
for name, runner in strategy_runners.items():
    produced = []
    for work_id, units in units_by_work.items():
        produced.extend(runner(work_id, work_titles[work_id], units))
    strategies[name] = produced

print(f'Tokenizer: {TOKENIZER}; budget {TARGET_TOKENS} tokens, overlap {OVERLAP_TOKENS}.')
for name, chunks in strategies.items():
    print(f'{name:>16}: {len(chunks):,} chunks')  # :>16 right-aligns the name in a 16-char column


### 3b. Model-Assisted Boundaries (Optional, Cost-Bounded)

- **Idea:** instead of a rule, ask the chat deployment to propose semantic break points for one scene, so a long speech splits where a *topic* shifts rather than where a token budget runs out.
- **Scope on purpose:** one scene only. Model chunking has real latency and token cost; the goal is a reviewable sample, not to re-chunk the whole corpus expensively — the same discipline notebook 01 used for its model parser.
- **Runs offline-safe:** if `.env` or Azure credentials are unavailable, this step records that it was skipped and the notebook continues; the selection never depends on it.
- **Expected result:** the model's proposed boundary count with latency and token usage, or a clear "skipped" note. Either way it stays a reserved fallback, not the default.


In [ ]:
import time

# next(iter(dict)) grabs the first key; max(..., key=fn) picks the item scoring highest under fn.
sample_work_id = next(iter(units_by_work))
sample_scene_units = max(scene_groups(units_by_work[sample_work_id]),
                         key=lambda g: sum(u['tokens'] for u in g))  # the longest scene (most tokens)
sample_scene_text = '\n\n'.join(u['text'] for u in sample_scene_units)
sample_scene_label = (f"{work_titles[sample_work_id]} Act {sample_scene_units[0]['act']} "
                      f"Scene {sample_scene_units[0]['scene']}")

# Default record assumes the call is unavailable; it is overwritten on success and read by the export step.
model_assisted_note = {'attempted': True, 'available': False, 'scene': sample_scene_label}
try:
    from dotenv import dotenv_values  # reads KEY=VALUE pairs from a .env file into a dict
    from azure.identity import DefaultAzureCredential, get_bearer_token_provider
    from openai import AzureOpenAI

    # Keyless auth via Azure identity, matching notebook 00/01 — no account keys in the notebook.
    config = {key: value for key, value in dotenv_values(ROOT / '.env').items() if value}
    # get_bearer_token_provider returns a callable that hands the client a fresh auth token when needed.
    token_provider = get_bearer_token_provider(
        DefaultAzureCredential(), 'https://cognitiveservices.azure.com/.default',
    )
    openai_client = AzureOpenAI(  # the client object we call to reach the hosted model
        azure_endpoint=config['AZURE_OPENAI_ENDPOINT'],
        api_version=config['AZURE_OPENAI_API_VERSION'],
        azure_ad_token_provider=token_provider,
    )

    system_prompt = (
        'You segment one scene of a Shakespeare play into coherent retrieval chunks. '
        'Return a JSON object {"boundaries": [int, ...]} giving the line indices (0-based, into the '
        'provided numbered lines) where a new chunk should start. Break on topic or speaker shifts, '
        'aiming for roughly 512-token chunks. Do not rewrite or summarize the text.'
    )
    # Number each block so the model can reply with integer break points instead of rewriting text.
    numbered = '\n'.join(f'{i}: {u["text"]}' for i, u in enumerate(sample_scene_units))

    started = time.perf_counter()  # perf_counter() is a high-resolution timer for measuring elapsed seconds
    # chat.completions.create sends the prompt to the model; response_format asks for strict JSON back.
    response = openai_client.chat.completions.create(
        model=config['AZURE_OPENAI_CHAT_DEPLOYMENT'],
        messages=[{'role': 'system', 'content': system_prompt},
                  {'role': 'user', 'content': numbered}],
        response_format={'type': 'json_object'},
    )
    latency_ms = round((time.perf_counter() - started) * 1000)  # elapsed seconds -> milliseconds
    raw = response.choices[0].message.content or '{}'  # the model's text reply (the JSON string)
    match = re.search(r'\{.*\}', raw, re.DOTALL)  # find the {...} block; re.DOTALL lets . match newlines too
    boundaries = json.loads(match.group(0) if match else '{}').get('boundaries', [])  # parse it to a Python list
    usage = response.usage  # token counts the API reports for this call

    model_assisted_note.update({  # dict.update() merges these keys into the existing record
        'available': True,
        'proposed_boundaries': len(boundaries),
        'latency_ms': latency_ms,
        'total_tokens': usage.total_tokens,
    })
    print(f'Model-assisted sample: {sample_scene_label} ({len(sample_scene_units)} blocks)')
    print(f'Proposed boundaries: {len(boundaries)}; latency {latency_ms} ms; tokens {usage.total_tokens}')
    print('Reserved as a fallback — not applied to the full corpus here.')
except Exception as error:
    # Offline-safe: record why it was skipped and let the deterministic strategies stand on their own.
    model_assisted_note['skip_reason'] = type(error).__name__  # the exception's class name, e.g. 'KeyError'
    print(f'Model-assisted arm skipped ({type(error).__name__}); continuing with deterministic strategies.')
    print(f'Sample scene that would be used: {sample_scene_label}')


## Metrics in This Stage — What Each Number Means

Section 4 scores each chunking strategy on size distribution, structural integrity, and an answerability proxy. All are computed over the chunks each strategy produces.

| Metric | What it measures | How it is computed | What it signifies |
|---|---|---|---|
| Token distribution (p50 / p95 / max) | the sizes the embedding model will actually see | `statistics.quantiles()` over every chunk's token count | p50 and p95 near the 512-token budget means uniform, window-safe chunks; a large max flags an oversized block. |
| Tiny-chunk share (under 64 tokens) | context-poor fragments | share of chunks with `token_count < 64` | Precision risk (noisy matches); lower is better. |
| Oversized share (over 110% of budget) | window-overflow risk | share of chunks with `token_count > 512 * 1.1` | Truncation and signal loss; lower is better (target ≤ 10%). |
| Scene-crossing chunks | structural integrity | count of chunks whose spanned (act, scene) count is greater than one | A chunk blending two scenes breaks attribution; 0% is the goal for scene-aware chunking. |
| Evidence fragmentation (avg fragments, captured-whole %) | how hard a cited scene is to reassemble | per cited scene, the number of chunks it lands in; `statistics.mean(fragments)` and the share of cited scenes held in exactly one chunk | Fewer fragments means easier assembly downstream; higher captured-whole is better. |


## 4. Measure and Inspect

- **Three lenses, no free ground truth:** we score each strategy on size distribution, provenance integrity, and an answerability proxy, then review a sample by eye.
- **Size distribution:** we summarize the pile of per-chunk token counts. **p50** (median) is the typical size; **p95** is the size 95% of chunks stay under (the tail, ignoring one freak value); **max** is the single largest chunk. We also track **% tiny** (< 64 tokens — one-liners with too little content to retrieve) and **% oversized** (> 110% of budget — risk overflowing the embedding window and diluting precision). A tight distribution hugging 512 retrieves more predictably.
- **Provenance integrity:** the share of chunks that **cross a scene boundary**. The structure-aware strategies are 0% by construction; the windows are not — this is the core tradeoff.
- **Answerability proxy — a metadata lookup, not a text search:** each labeled question already cites *where* its answer lives (e.g. "Hamlet, Act 1, Scene 5"). We parse that citation into a `(work, act, scene)` coordinate and check which chunks carry that coordinate in their metadata — no embeddings, no keyword search of the question against the text. From the matches we report **average fragments** (how many chunks the scene is split across) and **captured-whole %** (the scene sits in a single chunk). *Diagnostic, not decisive*: because the labels are scene-level, "captured whole" trivially rewards giant chunks, so scene-aware scores 100% only because its chunks are oversized. Whether a chunk actually contains the answer *and ranks first* needs the real retriever in notebook 05.
- **Expected result:** one comparison table across strategies plus a sample chunk printed for inspection. Expect the windows to lose on integrity, scene-aware to win fragmentation but blow the size budget, and recursive to balance both.


In [ ]:
import statistics  # standard library for averages, medians, etc.

title_by_text = {work_id: title.lower() for work_id, title in work_titles.items()}  # lowercased titles for matching


def distribution(chunks):
    tokens = sorted(c['token_count'] for c in chunks)  # all chunk sizes, ascending
    n = max(len(chunks), 1)  # guard against divide-by-zero
    crossing = sum(1 for c in chunks if c['scene_count'] > 1)  # `sum(1 for ... if ...)` counts matches
    tiny = sum(1 for c in chunks if c['token_count'] < 64)
    oversize = sum(1 for c in chunks if c['token_count'] > TARGET_TOKENS * 1.1)
    # p95 = the size 95% of chunks stay at or below. We index into the sorted list at the 95th-percentile position.
    p95 = tokens[min(len(tokens) - 1, int(0.95 * len(tokens)))] if tokens else 0
    return {
        'chunks': len(chunks),
        'p50': int(statistics.median(tokens)) if tokens else 0,  # median = the middle value
        'p95': p95,
        'max': max(tokens) if tokens else 0,
        'tiny_pct': tiny / n,
        'oversize_pct': oversize / n,
        'scene_crossing_pct': crossing / n,
    }


def resolve_targets(question):
    # Turn a label like "In Hamlet ... Act 1, Scene 5" into (work_id, act, scene) coordinates.
    # This reads the label's citation only — it does NOT search the question's words against the text.
    text = (question.get('question', '') + ' ' + question.get('expected_evidence', '')).lower()
    works = [work_id for work_id, title in title_by_text.items() if title in text]  # play named in the label
    # re.findall returns every match of the pattern; here each match is an (act, scene) pair of captured groups.
    pairs = re.findall(r'act\s+(\d+|[ivxlc]+)\s*,?\s*scene\s+(\d+|[ivxlc]+)',
                       question.get('expected_evidence', '').lower())  # \s+ = spaces, \d+ = digits, (...)= captured
    acts_scenes = [(roman_to_int(a), roman_to_int(s)) for a, s in pairs]
    # Nested comprehension: pair every matched play with every matched (act, scene).
    return [(work_id, act, scene) for work_id in works for act, scene in acts_scenes]


def chunk_covers(chunk, work_id, act, scene):
    # True when the chunk is from the work and lists that (act, scene) in its metadata — a lookup, not a search.
    # any(...) is True if at least one stored (a, s) pair equals the target.
    return chunk['work_id'] == work_id and any(a == act and s == scene for a, s in chunk['scenes'])


def answerability(chunks):
    by_work = defaultdict(list)  # group chunks per play so we only scan the relevant ones
    for chunk in chunks:
        by_work[chunk['work_id']].append(chunk)
    resolved, whole, fragments = 0, 0, []
    for question in evaluation_questions.get('development', []):
        if not question.get('answerable'):  # skip unanswerable questions (no evidence scene to find)
            continue
        for work_id, act, scene in resolve_targets(question):
            if act is None or scene is None:
                continue
            # All chunks that hold the target scene: 1 = scene kept whole, more = scene fragmented.
            covering = [c for c in by_work.get(work_id, []) if chunk_covers(c, work_id, act, scene)]
            if not covering:
                continue  # scene reference did not resolve to parsed structure; reported as unresolved
            resolved += 1
            fragments.append(len(covering))
            if len(covering) == 1:
                whole += 1
    return {
        'resolved_targets': resolved,
        'captured_whole_pct': whole / resolved if resolved else 0.0,
        'avg_fragments': statistics.mean(fragments) if fragments else 0.0,  # mean = average
    }


# {**a, **b} merges two dicts: here each strategy's distribution stats plus its answerability stats.
candidates = {name: {**distribution(chunks), **answerability(chunks)}
              for name, chunks in strategies.items()}

header = ('strategy', 'chunks', 'p50', 'p95', 'max', 'tiny%', 'over%', 'cross%', 'whole%', 'frags')
# str.format with {:>5} etc. right-aligns each column to a fixed width so the table lines up.
print('{:>16} {:>7} {:>5} {:>5} {:>6} {:>6} {:>6} {:>7} {:>7} {:>6}'.format(*header))
for name, m in candidates.items():
    print('{:>16} {:>7} {:>5} {:>5} {:>6} {:>5.0%} {:>5.0%} {:>6.0%} {:>6.0%} {:>6.2f}'.format(  # .0% shows a percent
        name, m['chunks'], m['p50'], m['p95'], m['max'],
        m['tiny_pct'], m['oversize_pct'], m['scene_crossing_pct'],
        m['captured_whole_pct'], m['avg_fragments']))

resolved_any = next((m['resolved_targets'] for m in candidates.values()), 0)  # first value, or 0 if none
print(f'\nAnswerability resolved {resolved_any} labeled evidence scene(s) from the development set.')

# Inspect one structure-aware chunk by eye. next((x for x in ... if cond), default) returns the first match.
sample = next((c for c in strategies['recursive-512-64'] if c['speakers']), strategies['recursive-512-64'][0])
print(f"\nSample chunk [{sample['chunk_id']}] — {sample['work']} Act {sample['act']} Scene {sample['scene']} "
      f"({sample['token_count']} tokens, speakers: {', '.join(sample['speakers']) or 'none'})")
print(sample['text'][:400] + ('…' if len(sample['text']) > 400 else ''))  # [:400] = first 400 characters


## 5. Justify the Selection

- **Decision rule (data-driven, deterministic):** among the strategies, prefer the one that first **never crosses a scene boundary**, then **fits the token budget** (≤ 10% oversized chunks), then keeps **p95 tokens closest to 512**, and finally **fragments the labeled evidence least**. The rule is applied to the measured numbers, not asserted.
- **Why size discipline gates before fragmentation:** keeping whole scenes together (scene-aware) scores perfectly on the fragment proxy, but only because its chunks run to thousands of tokens — they overflow the embedding window and dilute retrieval precision. Gating on the size budget first prevents the proxy from selecting oversized chunks, which is the failure the markdown above warned about.
- **Expected outcome:** the structure-blind windows are rejected on integrity (they cross scene boundaries); scene-aware is rejected on size (most chunks are oversized); **recursive packing** wins — it never crosses a scene, stays near the 512-token budget, and still carries full provenance. The remaining evidence fragmentation is recovered downstream by overlap and retrieval (notebook 05).
- **Model-assisted stays reserved:** exactly as notebook 01 reserved the model parser as a coverage-triggered fallback, model-assisted chunking is kept for irregular passages, not made the default — its cost is not justified when a deterministic rule meets the constraints.
- **Expected result:** the selected strategy name with the concrete numbers that justify it, and the rejected options noted.


In [ ]:
def selection_score(metrics):
    # Returns a tuple that Python compares left-to-right, so earlier items dominate (lexicographic order).
    # Lower is better. Integrity first; then the chunk must fit the token budget
    # (whole-scene capture alone trivially favors oversized chunks); then size fit
    # and least fragmentation among budget-respecting candidates.
    fits_budget = metrics['oversize_pct'] <= 0.10
    return (
        metrics['scene_crossing_pct'],              # 1st: fewest scene crossings
        0 if fits_budget else 1,                    # 2nd: must fit the budget (0 beats 1)
        abs(metrics['p95'] - TARGET_TOKENS),        # 3rd: p95 closest to 512 (abs() = distance)
        metrics['avg_fragments'] if metrics['avg_fragments'] else float('inf'),  # 4th: least fragmentation
    )


# min(items, key=fn) returns the item with the smallest key; here the best-scoring strategy name.
selected_strategy = min(candidates, key=lambda name: selection_score(candidates[name]))
selected = candidates[selected_strategy]
rejected = [name for name in candidates if name != selected_strategy]

print(f'Selected strategy: {selected_strategy}')
print(f"  scene crossings:   {selected['scene_crossing_pct']:.0%}")
print(f"  oversize chunks:   {selected['oversize_pct']:.0%} (budget {TARGET_TOKENS} tokens)")
print(f"  token p50/p95/max: {selected['p50']}/{selected['p95']}/{selected['max']}")
print(f"  evidence spread:   {selected['avg_fragments']:.2f} chunks per labeled scene "
      f"({selected['captured_whole_pct']:.0%} captured whole)")
print(f"  chunk count:       {selected['chunks']:,}")
print('\nRejected:')
for name in rejected:
    m = candidates[name]
    print(f"  {name}: cross={m['scene_crossing_pct']:.0%} oversize={m['oversize_pct']:.0%} "
          f"p95={m['p95']} frags={m['avg_fragments']:.2f}")


## 6. Export the Handoff

- **What gets exported:**
  - `.artifacts/chunks.json` — the selected chunks with text and full provenance metadata (work, act, scene, speakers, source-line span, token count), the input for notebook 03 (embeddings).
  - `.artifacts/chunking-handoff.json` — the decision record: tokenizer, token budget and overlap, the selected strategy, the full candidate comparison, chunk count, and a reference back to the Step 01 parsing handoff.
- **Provenance preserved:** each chunk still points back to the original source lines, so embeddings, retrieval, and labels downstream can trace evidence to the text.
- **Hygiene:** `.artifacts/` is ignored and notebook outputs are stripped on commit; do not commit chunks or artifacts.
- **Expected result:** both artifacts written, with the selected strategy and total chunk count printed.


In [ ]:
# Export the winning chunks and a decision record so notebook 03 (embeddings) starts
# from frozen artifacts, not hidden kernel state — the same handoff discipline as notebook 01.
ARTIFACTS.mkdir(exist_ok=True)  # create .artifacts/ if it does not exist yet
selected_chunks = strategies[selected_strategy]  # the chunks produced by the chosen strategy
chunks_path = ARTIFACTS / 'chunks.json'
chunking_handoff_path = ARTIFACTS / 'chunking-handoff.json'

# chunks.json: the selected chunks with text + full provenance — the input notebook 03 embeds.
# ensure_ascii=False keeps Shakespeare's accented characters readable instead of \u escapes.
chunks_path.write_text(json.dumps(selected_chunks, ensure_ascii=False), encoding='utf-8')

# chunking-handoff.json: the decision record — knobs, the winner, the full comparison, and a
# back-reference to notebook 01's parsing handoff so the provenance chain stays explicit.
chunking_handoff = {
    'stage': '02-chunking',
    'corpus_prefix': corpus_prefix,
    'tokenizer': TOKENIZER,
    'target_tokens': TARGET_TOKENS,
    'overlap_tokens': OVERLAP_TOKENS,
    'selected_strategy': selected_strategy,
    'chunk_count': len(selected_chunks),
    'candidates': candidates,                       # the measured comparison across all strategies
    'model_assisted': model_assisted_note,          # the reserved fallback's demo record (or skip note)
    'source_parsing_handoff': parsing_handoff.get('stage', 'parsing-handoff'),
}
chunking_handoff_path.write_text(json.dumps(chunking_handoff, ensure_ascii=False, indent=2), encoding='utf-8')

# .relative_to(ROOT) prints the short in-repo path instead of the full absolute one.
print(f'Wrote {len(selected_chunks):,} "{selected_strategy}" chunks to {chunks_path.relative_to(ROOT)}')
print(f'Wrote decision record to {chunking_handoff_path.relative_to(ROOT)}')
print('Reminder: .artifacts/ is gitignored and notebook outputs are stripped on commit — do not commit chunks.')


## Observations and Open Questions

These are honest notes from working through this stage — what the numbers do and do not prove, and what to revisit later.

- **The metrics here are proxies, not the real test.** Size distribution and scene-boundary integrity are measured directly, but "does this chunking help the final RAG answer correctly?" cannot be answered yet. The true test is end-to-end: retrieval (notebook 05), grounded generation (08), and continuous evaluation (10). Today's choice is a well-reasoned **default**, not a verdict.

- **The answerability proxy is weak by design — diagnostic only.**
  - *Presence is trivial:* every strategy chops the whole play into chunks, so every scene exists somewhere in every strategy's output. The coordinate always matches.
  - *What we actually read is fragmentation:* the **count** of chunks a labeled scene is spread across, not a pass/fail. That is the only real signal here.
  - *It rewards the wrong thing:* because labels are scene-level, "captured whole" trivially favors oversized chunks (scene-aware scores 100% just by being huge). So it must **not** decide the winner.
  - *The target resolution is crude:* a question citing two plays and two scenes produces a cross-product of `(work, act, scene)` targets, some spurious. They usually resolve to nothing and are dropped, but it is imprecise.
  - **Consequence:** the selection is driven by **integrity (no scene crossing) + size discipline**, with answerability kept only as a fragmentation diagnostic.

- **Chunk metadata is mostly a product requirement, with some experiment scaffolding.**
  - *Essential for a grounded evidence desk:* `work_id/work/act/scene/speakers/line_start/line_end` enable citations like "Hamlet, Act 1, Scene 1, Barnardo (lines 78–178)" — you cannot reconstruct those from chunk text alone.
  - *Scaffolding:* `scenes`, `scene_count`, and `strategy` exist mainly to keep this notebook's measurement code short; they add little to the downstream retrieval payload. A future cleanup could compute them transiently during measurement instead of baking them into every exported chunk.
  - *Real complexity lives in the algorithms*, not the metadata — the four boundary rules and the `owner` token-map are the hard part regardless of which fields a chunk carries.

- **How we will revisit (and the discipline for it).** Chunking is a swappable knob: change the selected strategy or the 512/64 budget, re-export `chunks.json`, and re-run 03→05→08 on the **same** questions. Iterate against the **development** set; check the **held-out** set only periodically to confirm gains generalize, so we do not slowly overfit the evaluation. Chunking is one of the highest-leverage things to retune once retrieval and generation are live.


## Completion Check and Next Notebook

- **Restart and run in order:** restart the kernel and run top to bottom to confirm this notebook depends only on the notebook 01 artifacts, not hidden state. It runs offline except for the optional model-assisted arm.
- **What we achieved:** four chunking strategies compared on size, provenance integrity, and an answerability proxy against the labeled scenes; a data-driven selection; a reserved model-assisted fallback; and two exported artifacts (`chunks.json`, `chunking-handoff.json`).
- **Honest limits:** the answerability proxy rewards keeping a scene's evidence in a single chunk — it is not retrieval accuracy, which is measured in notebook 05. Scene references that do not resolve to parsed structure are reported as unresolved rather than counted.
- **Privacy and hygiene:** outputs are stripped on commit by `nbstripout`; `.artifacts/` is ignored. Do not commit the chunks or decision record.
- **Next stage — 03 Embeddings:** load `.artifacts/chunks.json`, embed each chunk's text with the hosted embedding model, and compare embedding choices while holding the chunking selection fixed.


## Questions, Discussions, and Further Insights

The "Observations and Open Questions" cell above records honest caveats about *our* measurements. This section answers the broader conceptual questions a student should be able to reason about after this stage.

**Q1. Why 512 tokens and 64 overlap — are these magic numbers?**
No. 512 tokens (~350–400 words) is a widely used default: large enough to hold a coherent passage with context, small enough to retrieve precisely, and comfortably inside the embedding model's input window. 64 (~12% of 512) is a modest overlap that glues straddling sentences without heavily duplicating text. They are **starting points chosen by reasoning, not by experiment** — and the notebook is explicit that they should be retuned once retrieval (05) can measure their real effect. Treat any blog's "best" chunk size with suspicion; the best size is corpus- and query-dependent.

**Q2. The scene-aware strategy scored 100% on "captured whole" — why isn't it the winner?**
Because that proxy is trivially gamed by making chunks huge. A whole scene can run to thousands of tokens, which overflows the embedding window and dilutes retrieval precision — exactly the "chunks too large" failure. This is why the selection rule gates on **size discipline first** and uses fragmentation only as a tiebreaker. It's a clean illustration of a general lesson: a metric that can be maximized the wrong way must never be the sole decision criterion.

**Q3. Isn't overlap just wasteful duplication?**
It is a deliberate trade. Overlap costs index size and some redundant text, and in return it keeps boundary-straddling sentences readable and improves recall near cuts. For *structure-aware* chunking the benefit is smaller (boundaries already fall on clean seams), which is why we use a modest 64 rather than a large overlap. For *structure-blind* windows, overlap is doing much more work because the cuts are arbitrary.

**Q4. When would semantic or model-assisted chunking actually be worth the cost?**
When the text has no reliable structure to exploit (free-form prose, transcripts, optical character recognition (OCR) output) *and* fixed/recursive windows demonstrably hurt retrieval quality. Semantic chunking cuts where meaning shifts, which can beat token-count cuts on such text — but it requires embedding every sentence (cost) and is harder to reproduce. Our corpus has strong structure, so a deterministic rule already captures the semantic boundaries for free; that is why the model arm stays a reserved, cost-bounded fallback.

**Q5. How does the chunking choice ripple into later stages?**
Directly and irreversibly within a run. Chunks are embedded as-is (03), indexed as-is (04), and retrieved as-is (05); the final answer can only cite whole chunks (08). Over-large chunks reduce retrieval precision and produce sprawling, hard-to-verify citations; fragmented chunks can make the answer un-retrievable. This is why chunking is called one of the highest-leverage knobs — and why it is deliberately swappable: change the strategy, re-export `chunks.json`, and re-run 03→08 on the same questions.

**Q6. What would a production chunking stage add that this notebook doesn't?**
Likely: adaptive sizing per document type, a parent-child/hierarchical scheme so retrieval can fetch a precise child but hand the model its larger parent for context, contextual enrichment (prepending a short document/section summary to each chunk before embedding), and continuous re-evaluation as the corpus and query mix evolve. Several of these are retrieval-time architectures and are therefore explored in notebook 05, not here.

**Q7. What did we produce, and what happens next?**
We exported `.artifacts/chunks.json` (the selected chunks with full provenance) and `.artifacts/chunking-handoff.json` (the decision record and candidate comparison). Notebook 03 (Embeddings) loads `chunks.json` and turns each chunk's text into a vector, holding this chunking selection fixed while it experiments with the embedding dimension.
